# ARIMA Forecast – LastFM Sessions

**Exercise 3 – ML Challenge**

Steps:
1. Load data and build sessions (gap > 20 min = new session)
2. Answer: Top 10 songs in the top 50 longest sessions by track count
3. Select top 1 user by number of sessions
4. Forecast the next 3 months of **number of sessions** using ARIMA(1,1,1)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


## 1. Load data

In [ ]:
df = pd.read_csv(
    '../data/userid-timestamp-artid-artname-traid-traname.tsv',
    sep='\t',
    header=None,
    names=['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname'],
    on_bad_lines='skip'
)

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values(['userid', 'timestamp']).reset_index(drop=True)
print(f'Total rows: {len(df):,}')
df.head()

## 2. Build sessions (gap > 20 min = new session)

In [ ]:
GAP = pd.Timedelta(minutes=20)

df['prev_ts'] = df.groupby('userid')['timestamp'].shift(1)
df['new_session'] = (
    (df['timestamp'] - df['prev_ts'] > GAP) | df['prev_ts'].isna()
)
df['session_id'] = df.groupby('userid')['new_session'].cumsum()
df['session_key'] = df['userid'].astype(str) + '_' + df['session_id'].astype(str)

print(f'Total sessions: {df["session_key"].nunique():,}')

## 3. Top 10 songs in the top 50 longest sessions by track count

In [ ]:
session_track_count = df.groupby('session_key').size().rename('track_count')
top50_sessions = session_track_count.nlargest(50).index

top10_songs = (
    df[df['session_key'].isin(top50_sessions)]
    .groupby('traname')
    .size()
    .rename('plays')
    .nlargest(10)
    .reset_index()
)

print('Top 10 songs in top 50 longest sessions:')
top10_songs

## 4. Top 1 user by number of sessions

In [ ]:
user_session_count = df.groupby('userid')['session_id'].nunique().rename('n_sessions')
top_user = user_session_count.idxmax()
print(f'Top user: {top_user}  —  {user_session_count[top_user]:,} sessions')

## 5. Aggregate: weekly number of sessions for the top user

In [ ]:
user_df = df[df['userid'] == top_user].copy()

# One row per session with its start date
session_starts = (
    user_df.groupby('session_id')['timestamp']
    .min()
    .dt.tz_localize(None)
    .rename('session_start')
    .reset_index()
    .set_index('session_start')
)

# Weekly session count
weekly = session_starts.resample('W')['session_id'].count().rename('n_sessions')
weekly = weekly[weekly > 0]

print(f'Date range: {weekly.index.min().date()} → {weekly.index.max().date()}')

weekly.plot(title=f'Weekly sessions – user {top_user}', figsize=(12, 4))
plt.ylabel('Sessions')
plt.tight_layout()
plt.show()

## 6. ARIMA(1,1,1) forecast – next 3 months (≈13 weeks)

In [ ]:
model  = ARIMA(weekly, order=(1, 1, 1))
result = model.fit()
print(result.summary())

In [ ]:
FORECAST_WEEKS = 13  # ~3 months

forecast = result.get_forecast(steps=FORECAST_WEEKS)
fc_mean  = forecast.predicted_mean
fc_ci    = forecast.conf_int()

fig, ax = plt.subplots(figsize=(14, 5))
weekly.plot(ax=ax, label='Historical', color='steelblue')
fc_mean.plot(ax=ax, label='Forecast', color='darkorange', linestyle='--')
ax.fill_between(
    fc_ci.index,
    fc_ci.iloc[:, 0],
    fc_ci.iloc[:, 1],
    color='darkorange', alpha=0.2, label='95% CI'
)
ax.set_title(f'ARIMA(1,1,1) – Weekly sessions forecast (next 3 months) – user {top_user}')
ax.set_ylabel('Sessions')
ax.legend()
plt.tight_layout()
plt.show()

print('\nForecast values:')
fc_mean.rename('predicted_sessions').to_frame().join(fc_ci).round(2)